In [ ]:
# ============================================
# Re-establish Snowflake Session
# ============================================
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("Session re-established!")
print("Session active: " + str(session is not None))

Getting Started with the EY Challenge
Welcome to the 2026 EY Data & AI Challenge!

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

Confirm setup of External Access Integration (EAI)

In [ ]:
%%sql -r dataframe_1
-- EY 2026 AI & Data Challenge
-- Snowflake Mandatory Setup Script

-- ------------------------------------------------------------
-- 1. Create Challenge Database
-- ------------------------------------------------------------
CREATE DATABASE IF NOT EXISTS EY_WATER_QUALITY;
USE DATABASE EY_WATER_QUALITY;

CREATE SCHEMA IF NOT EXISTS CHALLENGE;
USE SCHEMA CHALLENGE;

CREATE WAREHOUSE IF NOT EXISTS EY_WH
  WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 120
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE EY_WH;

CREATE OR REPLACE NETWORK RULE EY_EXTERNAL_APIS_RULE
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = (
    'earthengine.googleapis.com',
    'storage.googleapis.com',
    'oauth2.googleapis.com',
    'landsatlook.usgs.gov',
    'planetarycomputer.microsoft.com',
    'api.planetarycomputer.microsoft.com',
    'planetarycomputer.blob.core.windows.net',
    '*.blob.core.windows.net',
    '*.dfs.core.windows.net',
    'login.microsoftonline.com',
    'pypi.org',
    'pypi.python.org',
    'pythonhosted.org',
    'files.pythonhosted.org'
  );

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION EY_EXTERNAL_ACCESS
  ALLOWED_NETWORK_RULES = (EY_EXTERNAL_APIS_RULE)
  ENABLED = TRUE;

GRANT USAGE ON INTEGRATION EY_EXTERNAL_ACCESS TO ROLE PUBLIC;
GRANT USAGE ON DATABASE EY_WATER_QUALITY TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA EY_WATER_QUALITY.CHALLENGE TO ROLE PUBLIC;
GRANT USAGE ON WAREHOUSE EY_WH TO ROLE PUBLIC;

SHOW INTEGRATIONS LIKE 'EY_EXTERNAL_ACCESS';
SHOW NETWORK RULES LIKE 'EY_EXTERNAL_APIS_RULE';
SHOW DATABASES LIKE 'EY_WATER_QUALITY';
SHOW WAREHOUSES LIKE 'EY_WH';

Import Python packages

In [ ]:
!pip install requests

Load Python Dependencies

In [ ]:
with open("requirements.txt", "w") as f:
    f.write("pandas\nnumpy\nscikit-learn\nxgboost\nrequests\n")

In [ ]:
!pip install -r requirements.txt

In [ ]:
# ============================================
# Install Missing Packages
# ============================================
import subprocess
import sys

packages = [
    "pystac==1.11.0",
    "pystac_client==0.9.0", 
    "planetary_computer==1.0.0",
    "odc-stac==0.3.10",
    "rioxarray==0.17.0",
    "rasterio==1.4.3",
    "shapely==2.1.1",
    "tqdm==4.66.5",
    "xarray==2025.3.1",
    "geopandas==1.1.1",
    "netCDF4==1.7.2",
    "adlfs==2025.8.0",
    "zarr==2.17.2",
    "dask==2024.10.0",
    "numcodecs==0.12.1",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {pkg} installed")
    else:
        print(f"  ❌ {pkg} FAILED: {result.stderr[-200:]}")

print("\n✅ All installations attempted!")
print("⚠️  RESTART KERNEL NOW before running the next cell!")


In [ ]:
# ============================================
# Setup: Import Libraries and Create Session
# ============================================

# Snowflake session - ALWAYS FIRST
import snowflake
from snowflake.snowpark.context import get_active_session

# Create session immediately before anything else
session = get_active_session()
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("Session active: " + str(session is not None))
print("Database: EY_WATER_QUALITY")
print("Schema: CHALLENGE")

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Core libraries
import os
from datetime import date
from tqdm import tqdm
import numpy as np
import pandas as pd
from IPython.display import display

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Geo - REMOVED xarray import (causes numpy conflict)
# import xarray as xr  
from scipy.spatial import cKDTree

# ML - preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score

# ML - models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ML - metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Planetary Computer / STAC
try:
    import pystac_client
    import planetary_computer as pc
    from odc.stac import stac_load
    from pystac.extensions.eo import EOExtension as eo
    print("Planetary Computer libraries loaded!")
except Exception as e:
    print("Planetary Computer libraries not available: " + str(e))
    print("Continuing without them - not needed for model training!")

print("Libraries imported successfully.")
print("Session active: " + str(session is not None))

Response Variable

In [ ]:
# ============================================
# Set Database and Schema Context
# ============================================
print("=" * 80)
print("🔧 SETTING SNOWFLAKE CONTEXT")
print("=" * 80)

# Set the database and schema
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("\n✅ Context set:")
print(f"   Database: EY_WATER_QUALITY")
print(f"   Schema: CHALLENGE")
print("=" * 80)

Response Variable

In [ ]:
# ============================================
# Load all training datasets from Snowflake tables
# ============================================
print("\n📊 Loading data from Snowflake tables...")

# Ensure we're in the correct database and schema
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

# Water quality training data (target variables)
Water_Quality_df = session.table("WATER_QUALITY_TRAINING").to_pandas()
print(f"✅ Water Quality Training: {Water_Quality_df.shape[0]:,} rows × {Water_Quality_df.shape[1]} columns")

# Landsat features (satellite data)
Landsat_Features_df = session.table("LANDSAT_FEATURES_TRAINING").to_pandas()
print(f"✅ Landsat Features Training: {Landsat_Features_df.shape[0]:,} rows × {Landsat_Features_df.shape[1]} columns")

# TerraClimate features (climate data)
TerraClimate_Features_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()
print(f"✅ TerraClimate Features Training: {TerraClimate_Features_df.shape[0]:,} rows × {TerraClimate_Features_df.shape[1]} columns")

# Validation datasets
Landsat_Features_val_df = session.table("LANDSAT_FEATURES_VALIDATION").to_pandas()
print(f"✅ Landsat Features Validation: {Landsat_Features_val_df.shape[0]:,} rows × {Landsat_Features_val_df.shape[1]} columns")

TerraClimate_Features_val_df = session.table("TERRACLIMATE_FEATURES_VALIDATION").to_pandas()
print(f"✅ TerraClimate Features Validation: {TerraClimate_Features_val_df.shape[0]:,} rows × {TerraClimate_Features_val_df.shape[1]} columns")

print("\n✨ All datasets loaded successfully!")

Predictor Variables

In [ ]:
# ============================================
# Feature Engineering - Additional Features
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("🔧 FEATURE ENGINEERING - ADDING NEW FEATURES")
print("=" * 80)

# -------------------------------------------------------
# 1. DERIVED SPECTRAL INDICES
# -------------------------------------------------------
print("\n📡 Computing Spectral Indices...")

# NDWI - Normalized Difference Water Index
# Formula: (GREEN - NIR) / (GREEN + NIR)
# Highlights open water, suppresses vegetation/soil noise
landsat_train_features['NDWI'] = (
    (landsat_train_features['GREEN'] - landsat_train_features['NIR']) /
    (landsat_train_features['GREEN'] + landsat_train_features['NIR'])
)
print("   NDWI computed (GREEN - NIR) / (GREEN + NIR)")

# NIR_SWIR_RATIO - Turbidity/sediment indicator
# Formula: NIR / SWIR22
# Higher values indicate clearer water
landsat_train_features['NIR_SWIR_RATIO'] = (
    landsat_train_features['NIR'] / landsat_train_features['SWIR22'].replace(0, np.nan)
)
print("  NIR_SWIR_RATIO computed (NIR / SWIR22)")

# SWIR_RATIO - Surface moisture indicator  
# Formula: SWIR16 / SWIR22
landsat_train_features['SWIR_RATIO'] = (
    landsat_train_features['SWIR16'] / landsat_train_features['SWIR22'].replace(0, np.nan)
)
print("  SWIR_RATIO computed (SWIR16 / SWIR22)")

# -------------------------------------------------------
# 2. TEMPORAL FEATURES
# -------------------------------------------------------
print("\n Extracting Temporal Features...")

# Convert SAMPLE_DATE to datetime
landsat_train_features['SAMPLE_DATE'] = pd.to_datetime(landsat_train_features['SAMPLE_DATE'], dayfirst=True)

# Extract month (1-12) — captures seasonal variation
landsat_train_features['MONTH'] = landsat_train_features['SAMPLE_DATE'].dt.month
print("  MONTH extracted (1-12)")

# Extract year
landsat_train_features['YEAR'] = landsat_train_features['SAMPLE_DATE'].dt.year
print("  YEAR extracted")

# Extract season
# Summer: Dec-Feb, Autumn: Mar-May, Winter: Jun-Aug, Spring: Sep-Nov (Southern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 1  # Summer
    elif month in [3, 4, 5]:
        return 2  # Autumn
    elif month in [6, 7, 8]:
        return 3  # Winter
    else:
        return 4  # Spring

landsat_train_features['SEASON'] = landsat_train_features['MONTH'].apply(get_season)
print("   SEASON extracted (Southern Hemisphere: 1=Summer, 2=Autumn, 3=Winter, 4=Spring)")

# -------------------------------------------------------
# 3. SUMMARY
# -------------------------------------------------------
print("\n" + "=" * 80)
print(" UPDATED FEATURE SET")
print("=" * 80)
print(f"\nTotal columns: {landsat_train_features.shape[1]}")
print(f"Total rows: {landsat_train_features.shape[0]:,}")
print("\nAll columns:")
for i, col in enumerate(landsat_train_features.columns, 1):
    print(f"  {i}. {col}")

print("\nSample data (first 3 rows):")
display(landsat_train_features.head(3))

# Check for any NaN values introduced
nan_counts = landsat_train_features[['NDWI','NIR_SWIR_RATIO','SWIR_RATIO','MONTH','YEAR','SEASON']].isna().sum()
print("\n⚠️  NaN check on new features:")
print(nan_counts)
print("\n Feature engineering complete!")
print("=" * 80)

In [ ]:
# ============================================
# Data Type Conversion - All Features
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("🔧 DATA TYPE CONVERSION - ALL FEATURES")
print("=" * 80)

# -------------------------------------------------------
# 1. Original spectral indices
# -------------------------------------------------------
print("\n📡 Converting original spectral indices...")

for col in ['NDMI', 'MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = landsat_train_features[col].astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 2. New spectral indices
# -------------------------------------------------------
print("\n📡 Converting new spectral indices...")

for col in ['NDWI', 'NIR_SWIR_RATIO', 'SWIR_RATIO']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(
            landsat_train_features[col], errors='coerce'
        ).astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 3. Temporal features
# -------------------------------------------------------
print("\n📅 Converting temporal features...")

for col in ['MONTH', 'YEAR', 'SEASON']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = landsat_train_features[col].astype(int)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 4. Core spectral bands
# -------------------------------------------------------
print("\n🛰️  Converting core spectral bands...")

for col in ['NIR', 'GREEN', 'SWIR16', 'SWIR22']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(
            landsat_train_features[col], errors='coerce'
        ).astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 5. Handle any NaN values introduced by coercion
# -------------------------------------------------------
print("\n🔧 Checking for NaN values after conversion...")
nan_counts = landsat_train_features.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

if len(nan_cols) > 0:
    print(f"  ⚠️  Found NaNs in: {nan_cols.to_dict()}")
    print("  🔄 Filling NaNs with column medians...")
    landsat_train_features = landsat_train_features.fillna(
        landsat_train_features.median(numeric_only=True)
    )
    print("  ✅ NaNs filled with median values")
else:
    print("  ✅ No NaN values found — data is clean!")

# -------------------------------------------------------
# 6. Summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("📋 ALL COLUMN DATA TYPES AFTER CONVERSION:")
print("=" * 80)
print(landsat_train_features.dtypes)

print("\n" + "=" * 80)
print("📊 FINAL DATASET INFO:")
print("=" * 80)
print(f"  Rows:    {landsat_train_features.shape[0]:,}")
print(f"  Columns: {landsat_train_features.shape[1]}")

print("\nSample data (first 5 rows):")
display(landsat_train_features.head(5))

print("\n" + "=" * 80)
print("✅ ALL DATA TYPE CONVERSIONS COMPLETE!")
print("=" * 80)

Loading Pre-Extracted TerraClimate Data

In [ ]:
# ============================================
# Loading Pre-Extracted TerraClimate Data
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("LOADING TERRACLIMATE DATA")
print("=" * 80)

# Load TerraClimate features from Snowflake table
Terraclimate_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()

print(f"\nTerraClimate training features loaded: {Terraclimate_df.shape[0]:,} rows x {Terraclimate_df.shape[1]} columns")

print("\nTerraClimate feature columns:")
print(Terraclimate_df.columns.tolist())

print("\nSample data (first 5 rows):")
display(Terraclimate_df.head(5))

print("\n" + "=" * 80)
print("Key TerraClimate Features:")
print("=" * 80)
print("\n  PET (Potential Evapotranspiration) - Represents the atmospheric demand for")
print("  moisture, capturing climatic conditions such as temperature, humidity, and")
print("  radiation that influence surface water evaporation and affect water quality.")

# -------------------------------------------------------
# Convert PET to numeric and handle NaNs
# -------------------------------------------------------
print("\nConverting data types...")

if 'PET' in Terraclimate_df.columns:
    Terraclimate_df['PET'] = pd.to_numeric(Terraclimate_df['PET'], errors='coerce')
    print(f"  PET converted to float: {Terraclimate_df['PET'].dtype}")

# Convert any remaining object columns to numeric
for col in Terraclimate_df.select_dtypes(include=['object']).columns:
    if 'DATE' not in col.upper():
        Terraclimate_df[col] = pd.to_numeric(Terraclimate_df[col], errors='coerce')
        print(f"  {col} converted to numeric")

# Handle NaN values
nan_count = Terraclimate_df.isna().sum().sum()
if nan_count > 0:
    print(f"\n  Found {nan_count} NaN values - filling with median...")
    Terraclimate_df = Terraclimate_df.fillna(Terraclimate_df.median(numeric_only=True))
    print("  NaNs filled with median values")
else:
    print("  No NaN values found - data is clean!")

# -------------------------------------------------------
# Summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("TERRACLIMATE DATA SUMMARY:")
print("=" * 80)
display(Terraclimate_df.describe())

print("\n" + "=" * 80)
print("TerraClimate data loading complete!")
print("=" * 80)

In [ ]:
# ============================================
# TerraClimate Feature Engineering
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("TERRACLIMATE FEATURE ENGINEERING")
print("=" * 80)

# Load TerraClimate features from Snowflake table
Terraclimate_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()

# -------------------------------------------------------
# 1. Convert data types
# -------------------------------------------------------
print("\nConverting data types...")
Terraclimate_df['PET'] = pd.to_numeric(Terraclimate_df['PET'], errors='coerce')
Terraclimate_df['SAMPLE_DATE'] = pd.to_datetime(Terraclimate_df['SAMPLE_DATE'], dayfirst=True)
print("  PET converted to float")
print("  SAMPLE_DATE converted to datetime")

# -------------------------------------------------------
# 2. PET derived features
# -------------------------------------------------------
print("\nEngineering PET-derived features...")

Terraclimate_df['PET_SQUARED'] = Terraclimate_df['PET'] ** 2
print("  PET_SQUARED computed")

Terraclimate_df['PET_LOG'] = np.log1p(Terraclimate_df['PET'].clip(lower=0))
print("  PET_LOG computed")

pet_monthly_mean = Terraclimate_df.groupby(
    Terraclimate_df['SAMPLE_DATE'].dt.month
)['PET'].transform('mean')
Terraclimate_df['PET_MONTHLY_ANOMALY'] = Terraclimate_df['PET'] - pet_monthly_mean
print("  PET_MONTHLY_ANOMALY computed")

# -------------------------------------------------------
# 3. Location-based features
# -------------------------------------------------------
print("\nEngineering location-based features...")

Terraclimate_df['LOC_CLUSTER'] = pd.cut(
    Terraclimate_df['LATITUDE'],
    bins=5,
    labels=[1, 2, 3, 4, 5]
).astype(float)
print("  LOC_CLUSTER computed")

# -------------------------------------------------------
# 4. Cyclical temporal features
# -------------------------------------------------------
print("\nExtracting cyclical temporal features...")

Terraclimate_df['TC_MONTH'] = Terraclimate_df['SAMPLE_DATE'].dt.month
Terraclimate_df['TC_YEAR'] = Terraclimate_df['SAMPLE_DATE'].dt.year

Terraclimate_df['MONTH_SIN'] = np.sin(2 * np.pi * Terraclimate_df['TC_MONTH'] / 12)
Terraclimate_df['MONTH_COS'] = np.cos(2 * np.pi * Terraclimate_df['TC_MONTH'] / 12)
print("  MONTH_SIN computed")
print("  MONTH_COS computed")

# -------------------------------------------------------
# 5. Handle NaN values
# -------------------------------------------------------
nan_count = Terraclimate_df.isna().sum().sum()
if nan_count > 0:
    print(f"\n  Found {nan_count} NaN values - filling with median...")
    Terraclimate_df = Terraclimate_df.fillna(Terraclimate_df.median(numeric_only=True))
    print("  NaNs filled")
else:
    print("\n  No NaN values - data is clean!")

# -------------------------------------------------------
# 6. Summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("UPDATED TERRACLIMATE FEATURE SET:")
print("=" * 80)
print(f"\nShape: {Terraclimate_df.shape[0]:,} rows x {Terraclimate_df.shape[1]} columns")
print("\nAll columns:")
for i, col in enumerate(Terraclimate_df.columns, 1):
    print(f"  {i}. {col}")

print("\nSample data (first 5 rows):")
display(Terraclimate_df.head(5))

print("\nSummary statistics:")
display(Terraclimate_df.describe())

print("\n" + "=" * 80)
print("TerraClimate feature engineering complete!")
print("=" * 80)

In [ ]:
# ============================================
# Pre-Join Inspection
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("PRE-JOIN INSPECTION")
print("=" * 80)

# Check shapes
print("\nDataset shapes:")
print(f"  Water_Quality_df:      {Water_Quality_df.shape}")
print(f"  landsat_train_features: {landsat_train_features.shape}")
print(f"  Terraclimate_df:        {Terraclimate_df.shape}")

# Check columns
print("\nWater Quality columns:")
print(Water_Quality_df.columns.tolist())

print("\nLandsat columns:")
print(landsat_train_features.columns.tolist())

print("\nTerraClimate columns:")
print(Terraclimate_df.columns.tolist())

# Check for duplicate columns between datasets
landsat_cols = set(landsat_train_features.columns)
terra_cols = set(Terraclimate_df.columns)
wq_cols = set(Water_Quality_df.columns)

print("\nDuplicate columns between Landsat and TerraClimate:")
print(landsat_cols.intersection(terra_cols))

print("\nDuplicate columns between Landsat and Water Quality:")
print(landsat_cols.intersection(wq_cols))

print("\nDuplicate columns between TerraClimate and Water Quality:")
print(terra_cols.intersection(wq_cols))

# Check all rows match
print("\nRow count check:")
print(f"  All match: {Water_Quality_df.shape[0] == landsat_train_features.shape[0] == Terraclimate_df.shape[0]}")

Joining the Predictor Variables and Response Variables

In [ ]:
# ============================================
# MODEL BUILDING - FEATURE SELECTION
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

# ============================================
# Step 1: Check if wq_data exists
# ============================================
print("=" * 80)
print("MODEL BUILDING - FEATURE SELECTION")
print("=" * 80)

print("\nChecking if wq_data exists...")
try:
    print(f"wq_data found: {wq_data.shape}")
except NameError:
    print("wq_data not found. Recreating from Snowflake tables...")
    Water_Quality_df = session.table("WATER_QUALITY_TRAINING").to_pandas()
    landsat_train_features = session.table("LANDSAT_FEATURES_TRAINING").to_pandas()
    Terraclimate_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()
    wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
    wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
    print(f"wq_data recreated: {wq_data.shape}")

print("\n" + "=" * 80)

# ============================================
# Step 2: Show available columns
# ============================================
print("\nAvailable columns in wq_data:")
print(wq_data.columns.tolist())

print("\n" + "=" * 80)

# ============================================
# Step 3: Select required columns as per memo
# NOTE: Only SWIR22, NDMI, MNDWI from Landsat
# and PET from TerraClimate as predictors
# Latitude, Longitude and Sample Date excluded
# ============================================
print("\nSelecting predictor and target variables...")

required_cols = [
    # Predictor variables (as per memo)
    'SWIR22', 'NDMI', 'MNDWI', 'PET',
    # Target variables
    'TOTAL_ALKALINITY',
    'ELECTRICAL_CONDUCTANCE',
    'DISSOLVED_REACTIVE_PHOSPHORUS'
]

# Check which columns are available
available_cols = [col for col in required_cols if col in wq_data.columns]
missing_cols = [col for col in required_cols if col not in wq_data.columns]

if missing_cols:
    print(f"\nWARNING: Missing columns: {missing_cols}")
else:
    print("\nAll required columns found!")

# Select only required columns
wq_data = wq_data[available_cols]

print(f"\nFinal dataset shape: {wq_data.shape[0]:,} rows x {wq_data.shape[1]} columns")

print("\n" + "=" * 80)

# ============================================
# Step 4: Categorize columns
# ============================================
predictor_cols = ['SWIR22', 'NDMI', 'MNDWI', 'PET']
target_cols = [
    'TOTAL_ALKALINITY',
    'ELECTRICAL_CONDUCTANCE',
    'DISSOLVED_REACTIVE_PHOSPHORUS'
]

print("\nPREDICTOR VARIABLES (Features):")
print("-" * 40)
for i, col in enumerate(predictor_cols, 1):
    if col in wq_data.columns:
        print(f"  {i}. {col}: min={wq_data[col].min():.3f}, max={wq_data[col].max():.3f}, mean={wq_data[col].mean():.3f}")

print("\nTARGET VARIABLES (Response):")
print("-" * 40)
for i, col in enumerate(target_cols, 1):
    if col in wq_data.columns:
        print(f"  {i}. {col}: min={wq_data[col].min():.3f}, max={wq_data[col].max():.3f}, mean={wq_data[col].mean():.3f}")

print("\n" + "=" * 80)

# ============================================
# Step 5: Display sample and summary
# ============================================
print("\nSample of selected dataset (first 5 rows):")
print("=" * 80)
display(wq_data.head())

print("\n" + "=" * 80)
print("Summary statistics:")
print("=" * 80)
display(wq_data.describe())

print("\n" + "=" * 80)
print("DATA IS READY FOR TRAIN/TEST SPLIT AND MODEL TRAINING!")
print(f"  Predictor features: {len(predictor_cols)}")
print(f"  Target variables:   {len(target_cols)}")
print(f"  Total rows:         {wq_data.shape[0]:,}")
print("=" * 80)

In [ ]:
# ============================================
# Joining Predictor and Response Variables
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("JOINING PREDICTOR AND RESPONSE VARIABLES")
print("=" * 80)
print("\nCombining ground data (water quality) and predictor data (Landsat + TerraClimate)")
print("into a single dataset for model training.")
print("\n" + "=" * 80)

# Combining ground data and final data into a single dataset
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

print(f"\nCombined dataset created: {wq_data.shape[0]:,} rows x {wq_data.shape[1]} columns")

# Verify column count
print("\nAll columns in combined dataset:")
for i, col in enumerate(wq_data.columns, 1):
    print(f"  {i}. {col}")

# Verify no missing values
nan_total = wq_data.isna().sum().sum()
print(f"\nNaN check: {nan_total} missing values")

print("\n" + "=" * 80)
print("Sample of combined dataset (first 5 rows):")
print("=" * 80)
display(wq_data.head(5))

print("\n" + "=" * 80)
print("Dataset joining complete!")
print("=" * 80)

print("\nThe combined dataset now contains:")
print("\n  TARGET VARIABLES (3):")
print("    Total Alkalinity (TA)")
print("    Electrical Conductance (EC)")
print("    Dissolved Reactive Phosphorus (DRP)")

print("\n  LANDSAT SPECTRAL BANDS (4):")
print("    NIR, GREEN, SWIR16, SWIR22")

print("\n  LANDSAT SPECTRAL INDICES (5):")
print("    NDMI, MNDWI, NDWI, NIR_SWIR_RATIO, SWIR_RATIO")

print("\n  TEMPORAL FEATURES (3):")
print("    MONTH, YEAR, SEASON")

print("\n  TERRACLIMATE FEATURES (7):")
print("    PET, PET_SQUARED, PET_LOG, PET_MONTHLY_ANOMALY")
print("    LOC_CLUSTER, MONTH_SIN, MONTH_COS")

print("\n  LOCATION/DATE (3):")
print("    LATITUDE, LONGITUDE, SAMPLE_DATE")

print("\n" + "=" * 80)
print("Summary statistics of target variables:")
print("=" * 80)
for col in ['TOTAL_ALKALINITY', 'ELECTRICAL_CONDUCTANCE', 'DISSOLVED_REACTIVE_PHOSPHORUS']:
    if col in wq_data.columns:
        print(f"\n  {col}:")
        print(f"    Min:  {wq_data[col].min():.3f}")
        print(f"    Max:  {wq_data[col].max():.3f}")
        print(f"    Mean: {wq_data[col].mean():.3f}")
        print(f"    Std:  {wq_data[col].std():.3f}")

print("\n" + "=" * 80)
print("This unified dataset is ready for:")
print("  1. Handling missing values")
print("  2. Train/test split")
print("  3. Feature scaling")
print("  4. Model training")
print("=" * 80)

Handling Missing Values

In [ ]:
# FULL RECOVERY CELL
from snowflake.snowpark.context import get_active_session
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

session = get_active_session()
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")
print("Session ready!")

Water_Quality_df       = session.table("WATER_QUALITY_TRAINING").to_pandas()
landsat_train_features = session.table("LANDSAT_FEATURES_TRAINING").to_pandas()
Terraclimate_df        = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()
print("Data loaded!")

for col in ['NIR','GREEN','SWIR16','SWIR22','NDMI','MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(landsat_train_features[col], errors='coerce')

landsat_train_features['SAMPLE_DATE'] = pd.to_datetime(landsat_train_features['SAMPLE_DATE'], dayfirst=True)
landsat_train_features['NDWI']          = (landsat_train_features['GREEN'] - landsat_train_features['NIR']) / (landsat_train_features['GREEN'] + landsat_train_features['NIR'])
landsat_train_features['NIR_SWIR_RATIO'] = landsat_train_features['NIR'] / landsat_train_features['SWIR22'].replace(0, np.nan)
landsat_train_features['SWIR_RATIO']     = landsat_train_features['SWIR16'] / landsat_train_features['SWIR22'].replace(0, np.nan)
landsat_train_features['MONTH']          = landsat_train_features['SAMPLE_DATE'].dt.month
landsat_train_features['YEAR']           = landsat_train_features['SAMPLE_DATE'].dt.year
landsat_train_features['MONTH_SIN']      = np.sin(2 * np.pi * landsat_train_features['MONTH'] / 12)
landsat_train_features['MONTH_COS']      = np.cos(2 * np.pi * landsat_train_features['MONTH'] / 12)

def get_season(month):
    if month in [12,1,2]:  return 1
    elif month in [3,4,5]: return 2
    elif month in [6,7,8]: return 3
    else:                  return 4

landsat_train_features['SEASON'] = landsat_train_features['MONTH'].apply(get_season)
print("Landsat features done!")

Terraclimate_df['PET']           = pd.to_numeric(Terraclimate_df['PET'], errors='coerce')
Terraclimate_df['SAMPLE_DATE']   = pd.to_datetime(Terraclimate_df['SAMPLE_DATE'], dayfirst=True)
Terraclimate_df['PET_SQUARED']   = Terraclimate_df['PET'] ** 2
Terraclimate_df['PET_LOG']       = np.log1p(Terraclimate_df['PET'].clip(lower=0))
pet_monthly_mean                 = Terraclimate_df.groupby(Terraclimate_df['SAMPLE_DATE'].dt.month)['PET'].transform('mean')
Terraclimate_df['PET_MONTHLY_ANOMALY'] = Terraclimate_df['PET'] - pet_monthly_mean
Terraclimate_df['LOC_CLUSTER']   = pd.cut(Terraclimate_df['LATITUDE'], bins=5, labels=[1,2,3,4,5]).astype(float)
print("TerraClimate features done!")

wq_targets = Water_Quality_df[['TOTAL_ALKALINITY','ELECTRICAL_CONDUCTANCE','DISSOLVED_REACTIVE_PHOSPHORUS']].copy()

landsat_selected = landsat_train_features[[
    'NIR','GREEN','SWIR16','SWIR22','NDMI','MNDWI','NDWI',
    'NIR_SWIR_RATIO','SWIR_RATIO','MONTH','YEAR','SEASON','MONTH_SIN','MONTH_COS'
]].copy()

terra_selected = Terraclimate_df[[
    'PET','PET_SQUARED','PET_LOG','PET_MONTHLY_ANOMALY','LOC_CLUSTER'
]].copy()

wq_data = pd.concat([wq_targets, landsat_selected, terra_selected], axis=1)
wq_data = wq_data.loc[:, ~wq_data.columns.duplicated()]
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
print("wq_data shape: " + str(wq_data.shape))

def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = RobustScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test), scaler


def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print("  " + dataset_name + " R2: " + str(round(float(r2),4)) + " | RMSE: " + str(round(float(rmse),4)) + " | MAE: " + str(round(float(mae),4)))
    return y_pred, r2, rmse


print("Helper functions defined!")

feature_cols = [
    'NIR','GREEN','SWIR16','SWIR22',
    'NDMI','MNDWI','NDWI',
    'NIR_SWIR_RATIO','SWIR_RATIO',
    'PET','PET_SQUARED','PET_LOG',
    'PET_MONTHLY_ANOMALY','LOC_CLUSTER',
    'MONTH','YEAR','SEASON',
    'MONTH_SIN','MONTH_COS'
]

X     = wq_data[[col for col in feature_cols if col in wq_data.columns]]
y_TA  = wq_data['TOTAL_ALKALINITY']
y_EC  = wq_data['ELECTRICAL_CONDUCTANCE']
y_DRP = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

print("\nTraining models - this may take 2-3 minutes...")
print("=" * 80)

model_TA,  scaler_TA,  results_TA  = run_pipeline(X, y_TA,  "Total Alkalinity")
model_EC,  scaler_EC,  results_EC  = run_pipeline(X, y_EC,  "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

all_results = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)

print("\n" + "=" * 80)
print("FINAL MODEL SUMMARY")
print("=" * 80)
display(all_results)
print("\nReady for submission!")
print("  model_TA,  scaler_TA")
print("  model_EC,  scaler_EC")
print("  model_DRP, scaler_DRP")

Model Building

In [ ]:
# ============================================
# MODEL BUILDING - FEATURE SELECTION
# ============================================

# ============================================
# Step 1: Check if wq_data exists, if not recreate it
# ============================================
print("\n🔍 Checking if wq_data exists...")

try:
    print(f"✓ wq_data found: {wq_data.shape}")
except NameError:
    print("⚠️  wq_data not found. Recreating from Snowflake tables...")
    
    # Load all three datasets from Snowflake
    print("\n📊 Loading datasets from Snowflake...")
    
    Water_Quality_df = session.table("WATER_QUALITY_TRAINING").to_pandas()
    print(f"  ✓ Water Quality: {Water_Quality_df.shape}")
    
    landsat_train_features = session.table("LANDSAT_FEATURES_TRAINING").to_pandas()
    print(f"  ✓ Landsat Features: {landsat_train_features.shape}")
    
    Terraclimate_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()
    print(f"  ✓ TerraClimate Features: {Terraclimate_df.shape}")
    
    # Combine datasets
    print("\n🔗 Combining datasets...")
    wq_data = pd.concat([Water_Quality_df, landsat_train_features, Terraclimate_df], axis=1)
    wq_data = wq_data.loc[:, ~wq_data.columns.duplicated()]
    
    # Fill missing values
    print("🔧 Handling missing values...")
    numeric_columns = wq_data.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        wq_data[col] = pd.to_numeric(wq_data[col], errors='coerce')
    wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
    
    print(f"✅ wq_data created: {wq_data.shape}")

print("\n" + "=" * 80)

# ============================================
# Step 2: Show available columns
# ============================================
print("\n📋 Available columns in wq_data:")
print(wq_data.columns.tolist())

print("\n" + "=" * 80)

# ============================================
# Step 3: Select required columns
# ============================================
print("\n🎯 Selecting predictor and target variables...")

required_cols = ['SWIR22', 'NDMI', 'MNDWI', 'PET', 
                'TOTAL_ALKALINITY', 'ELECTRICAL_CONDUCTANCE', 
                'DISSOLVED_REACTIVE_PHOSPHORUS']

# Check which columns are available
available_cols = [col for col in required_cols if col in wq_data.columns]
missing_cols = [col for col in required_cols if col not in wq_data.columns]

if missing_cols:
    print(f"\n⚠️  Warning: Missing columns: {missing_cols}")
    print("\n🔍 Searching for similar column names...")
    for missing in missing_cols:
        similar = [c for c in wq_data.columns 
                  if missing.lower() in c.lower() or c.lower() in missing.lower()]
        if similar:
            print(f"  {missing} → Found similar: {similar}")
            # Use the first similar column found
            if similar[0] not in available_cols:
                available_cols.append(similar[0])
                print(f"    ✓ Using '{similar[0]}' instead")

# Select only available columns
wq_data = wq_data[available_cols]

print(f"\n✅ Feature selection complete!")
print(f"\nFinal dataset shape: {wq_data.shape[0]:,} rows × {wq_data.shape[1]} columns")

print("\n" + "=" * 80)

# ============================================
# Step 4: Categorize columns
# ============================================
print("\n📋 Selected columns:")

predictor_cols = [col for col in wq_data.columns 
                 if col not in ['TOTAL_ALKALINITY', 'ELECTRICAL_CONDUCTANCE', 'DISSOLVED_REACTIVE_PHOSPHORUS']]

target_cols = [col for col in wq_data.columns 
              if col in ['TOTAL_ALKALINITY', 'ELECTRICAL_CONDUCTANCE', 'DISSOLVED_REACTIVE_PHOSPHORUS']]

print("\n🎯 Predictor Variables (Features):")
for i, col in enumerate(predictor_cols, 1):
    print(f"  {i}. {col}")

print("\n🎯 Target Variables (Response):")
for i, col in enumerate(target_cols, 1):
    print(f"  {i}. {col}")

print("\n" + "=" * 80)

# ============================================
# Step 5: Display sample and summary
# ============================================
print("\nSample of selected dataset (first 5 rows):")
print("=" * 80)
display(wq_data.head())

print("\n" + "=" * 80)
print("Summary statistics:")
print("=" * 80)
display(wq_data.describe())

print("\n" + "=" * 80)
print("✅ DATA IS READY FOR TRAIN/TEST SPLIT AND MODEL TRAINING!")
print("=" * 80)

Helper Functions


In [ ]:
# ============================================
# Helper Functions
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("HELPER FUNCTIONS")
print("=" * 80)

print("""
TIP 3 - FEATURE COMBINATIONS:
We are developing individual models for each water quality parameter
using a common set of features: SWIR22, NDMI, MNDWI, and PET.

Our improvements over the benchmark:
  - RobustScaler instead of StandardScaler
  - Tuned Random Forest hyperparameters
  - Added MAE as additional evaluation metric
  - Added 5-fold cross validation
""")

def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2     = r2_score(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    print(f"\n  {dataset_name} Evaluation:")
    print(f"    R2 Score: {r2:.4f}  (explains {r2*100:.1f}% of variance)")
    print(f"    RMSE:     {rmse:.4f}")
    print(f"    MAE:      {mae:.4f}")
    return y_pred, r2, rmse

def cross_validate_model(X, y, param_name="Parameter"):
    model  = RandomForestRegressor(
        n_estimators=300, max_depth=12,
        min_samples_split=20, min_samples_leaf=10,
        max_features=0.6, random_state=42, n_jobs=-1
    )
    scaler   = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    cv_scores = cross_val_score(
        model, X_scaled, y, cv=5, scoring='r2', n_jobs=-1
    )
    print(f"\n  {param_name} - 5-Fold Cross Validation:")
    print(f"    CV R2 scores: {[round(s, 4) for s in cv_scores]}")
    print(f"    Mean CV R2:   {cv_scores.mean():.4f}")
    print(f"    Std CV R2:    {cv_scores.std():.4f}")
    if cv_scores.std() < 0.05:
        print(f"    Stable model - low variance across folds!")
    else:
        print(f"    Warning - high variance across folds!")
    return cv_scores

print("All helper functions defined:")
print("  1. split_data()           - 70/30 train/test split")
print("  2. scale_data()           - RobustScaler")
print("  3. train_model()          - Tuned Random Forest")
print("  4. evaluate_model()       - R2, RMSE and MAE")
print("  5. cross_validate_model() - 5-fold cross validation")
print("=" * 80)

 Function 1: Train and Test Split
 Split_data() - Splits data into 70% train, 30% test

In [ ]:
# ============================================
# Train and Test Split
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("TRAIN AND TEST SPLIT")
print("=" * 80)

print("""
We split the data into 70% training and 30% test data.
Scikit-learn provides the train_test_split function.
The response variable is NOT included in X.
Latitude, Longitude and Sample Date are EXCLUDED
as they serve only as spatial and temporal references.
""")

# Define predictor variables (X) and targets (y)
predictor_cols = ['SWIR22', 'NDMI', 'MNDWI', 'PET']
target_cols    = [
    'TOTAL_ALKALINITY',
    'ELECTRICAL_CONDUCTANCE',
    'DISSOLVED_REACTIVE_PHOSPHORUS'
]

X     = wq_data[predictor_cols]
y_ta  = wq_data['TOTAL_ALKALINITY']
y_ec  = wq_data['ELECTRICAL_CONDUCTANCE']
y_drp = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

print(f"Feature matrix X shape: {X.shape}")
print(f"Predictor variables:    {predictor_cols}")

print("\nSplitting data (70% train | 30% test)...")

# Split for Total Alkalinity
X_train_ta,  X_test_ta,  y_train_ta,  y_test_ta  = split_data(X, y_ta)
# Split for Electrical Conductance
X_train_ec,  X_test_ec,  y_train_ec,  y_test_ec  = split_data(X, y_ec)
# Split for Dissolved Reactive Phosphorus
X_train_drp, X_test_drp, y_train_drp, y_test_drp = split_data(X, y_drp)

print("\nSplit results:")
print(f"  Total Alkalinity:")
print(f"    Train: {X_train_ta.shape[0]:,} samples | Test: {X_test_ta.shape[0]:,} samples")
print(f"  Electrical Conductance:")
print(f"    Train: {X_train_ec.shape[0]:,} samples | Test: {X_test_ec.shape[0]:,} samples")
print(f"  Dissolved Reactive Phosphorus:")
print(f"    Train: {X_train_drp.shape[0]:,} samples | Test: {X_test_drp.shape[0]:,} samples")

print("\n" + "=" * 80)
print("Train/Test split complete!")
print("=" * 80)

 Function 2: Feature Scaling
 Scale_data() - Scales features using StandardScaler

In [ ]:
# ============================================
# Feature Scaling
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("FEATURE SCALING")
print("=" * 80)

print("""
TIP 4 - PREPROCESSING IMPROVEMENT:
Benchmark uses StandardScaler (mean=0, std=1).
We use RobustScaler which is better for environmental data
because it uses median and IQR instead of mean and std.
This makes it resistant to outliers such as flood events
or drought periods that cause extreme water quality readings.

Feature value ranges that justify RobustScaler:
  SWIR22: 3,634  to 31,202  (very wide range)
  PET:    52.7   to 270.8   (climate extremes possible)
  NDMI:  -0.328  to  0.568  (normalized index)
  MNDWI: -0.300  to  0.591  (normalized index)
""")

# Scale for Total Alkalinity
print("Scaling features for Total Alkalinity...")
X_train_ta_scaled,  X_test_ta_scaled,  scaler_ta  = scale_data(X_train_ta,  X_test_ta)
print("  Total Alkalinity features scaled!")

# Scale for Electrical Conductance
print("Scaling features for Electrical Conductance...")
X_train_ec_scaled,  X_test_ec_scaled,  scaler_ec  = scale_data(X_train_ec,  X_test_ec)
print("  Electrical Conductance features scaled!")

# Scale for Dissolved Reactive Phosphorus
print("Scaling features for Dissolved Reactive Phosphorus...")
X_train_drp_scaled, X_test_drp_scaled, scaler_drp = scale_data(X_train_drp, X_test_drp)
print("  Dissolved Reactive Phosphorus features scaled!")

print("\nVerifying scaled data statistics...")
import pandas as pd
scaled_stats = pd.DataFrame(
    X_train_ta_scaled,
    columns=predictor_cols
).describe().round(4)
display(scaled_stats)

print("\n" + "=" * 80)
print("Feature scaling complete!")
print("  All features scaled using RobustScaler")
print("  fit_transform on train | transform only on test")
print("  No data leakage from test set!")
print("=" * 80)

 Function 3 :Model Training
 Function 3: train_model() - Trains Random Forest Regressor

In [ ]:
# ============================================
# Anti-Overfitting Check + Model Training
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("ANTI-OVERFITTING CHECK BEFORE TRAINING")
print("=" * 80)

print("""
Before training we run 5-fold cross validation on each target.
This tells us the expected test R2 before we commit to training.
A low std across folds means the model will generalize well.

What to look for:
  Mean CV R2 > 0.65  - good predictive power
  Std CV R2  < 0.05  - stable, consistent model
  Train/Test gap < 0.10 - no overfitting
""")

# Cross validate all three targets
print("Running 5-fold cross validation...")
print("-" * 60)
cv_ta  = cross_validate_model(X, y_ta,  "Total Alkalinity")
cv_ec  = cross_validate_model(X, y_ec,  "Electrical Conductance")
cv_drp = cross_validate_model(X, y_drp, "Dissolved Reactive Phosphorus")

print("\n" + "=" * 80)
print("Cross validation summary:")
print(f"  Total Alkalinity          - Mean R2: {cv_ta.mean():.4f}  Std: {cv_ta.std():.4f}")
print(f"  Electrical Conductance    - Mean R2: {cv_ec.mean():.4f}  Std: {cv_ec.std():.4f}")
print(f"  Dissolved React. Phosph.  - Mean R2: {cv_drp.mean():.4f} Std: {cv_drp.std():.4f}")

print("\n" + "=" * 80)
print("MODEL TRAINING")
print("=" * 80)

print("""
Now training three separate Random Forest models.
One model per water quality parameter.
Using tuned hyperparameters to prevent overfitting.
""")

# Train Model 1: Total Alkalinity
print("Training Model 1: Total Alkalinity...")
model_ta  = train_model(X_train_ta_scaled,  y_train_ta)
print(f"  Model trained with {model_ta.n_estimators} trees | max_depth={model_ta.max_depth}")

# Train Model 2: Electrical Conductance
print("\nTraining Model 2: Electrical Conductance...")
model_ec  = train_model(X_train_ec_scaled,  y_train_ec)
print(f"  Model trained with {model_ec.n_estimators} trees | max_depth={model_ec.max_depth}")

# Train Model 3: Dissolved Reactive Phosphorus
print("\nTraining Model 3: Dissolved Reactive Phosphorus...")
model_drp = train_model(X_train_drp_scaled, y_train_drp)
print(f"  Model trained with {model_drp.n_estimators} trees | max_depth={model_drp.max_depth}")

print("\n" + "=" * 80)
print("All three models trained successfully!")
print("  model_ta  - Total Alkalinity")
print("  model_ec  - Electrical Conductance")
print("  model_drp - Dissolved Reactive Phosphorus")
print("=" * 80)

 Function 4: Model Evaluation
 Function 4: evaluate_model() - Evaluates model with R² and RMSE

In [ ]:
# ============================================
# Model Evaluation
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

print("""
Each model is evaluated using:
  R2 Score - Measures variance explained (higher is better)
  RMSE     - Average magnitude of errors (lower is better)
  MAE      - Mean absolute error (lower is better)

We evaluate on BOTH train and test sets.
A large gap between train and test R2 indicates overfitting.
Target: Train/Test gap < 0.10
""")

# -------------------------------------------------------
# Model 1: Total Alkalinity
# -------------------------------------------------------
print("=" * 80)
print("MODEL 1: TOTAL ALKALINITY (TA)")
print("=" * 80)
y_pred_train_ta, r2_train_ta, rmse_train_ta = evaluate_model(
    model_ta, X_train_ta_scaled, y_train_ta, "Train"
)
y_pred_ta, r2_ta, rmse_ta = evaluate_model(
    model_ta, X_test_ta_scaled, y_test_ta, "Test"
)
gap_ta = r2_train_ta - r2_ta
print(f"\n  Train/Test R2 gap: {gap_ta:.4f}", 
      "- Good!" if gap_ta < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Model 2: Electrical Conductance
# -------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL 2: ELECTRICAL CONDUCTANCE (EC)")
print("=" * 80)
y_pred_train_ec, r2_train_ec, rmse_train_ec = evaluate_model(
    model_ec, X_train_ec_scaled, y_train_ec, "Train"
)
y_pred_ec, r2_ec, rmse_ec = evaluate_model(
    model_ec, X_test_ec_scaled, y_test_ec, "Test"
)
gap_ec = r2_train_ec - r2_ec
print(f"\n  Train/Test R2 gap: {gap_ec:.4f}",
      "- Good!" if gap_ec < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Model 3: Dissolved Reactive Phosphorus
# -------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL 3: DISSOLVED REACTIVE PHOSPHORUS (DRP)")
print("=" * 80)
y_pred_train_drp, r2_train_drp, rmse_train_drp = evaluate_model(
    model_drp, X_train_drp_scaled, y_train_drp, "Train"
)
y_pred_drp, r2_drp, rmse_drp = evaluate_model(
    model_drp, X_test_drp_scaled, y_test_drp, "Test"
)
gap_drp = r2_train_drp - r2_drp
print(f"\n  Train/Test R2 gap: {gap_drp:.4f}",
      "- Good!" if gap_drp < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Final Summary Table
# -------------------------------------------------------
print("\n" + "=" * 80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame({
    'Model': [
        'Total Alkalinity',
        'Electrical Conductance',
        'Dissolved Reactive Phosphorus'
    ],
    'R2 Train': [r2_train_ta,  r2_train_ec,  r2_train_drp],
    'R2 Test':  [r2_ta,        r2_ec,        r2_drp],
    'Gap':      [gap_ta,       gap_ec,       gap_drp],
    'RMSE':     [rmse_ta,      rmse_ec,      rmse_drp]
})
display(summary_df.round(4))

print("\nOverfitting Assessment:")
for _, row in summary_df.iterrows():
    status = "Good generalization!" if row['Gap'] < 0.10 else "Possible overfitting!"
    print(f"  {row['Model']}: gap={row['Gap']:.4f} - {status}")

print("\n" + "=" * 80)
print("MODEL EVALUATION COMPLETE!")
print("=" * 80)

Model Workflow (Pipeline)

In [ ]:
# ============================================
# Run Pipeline for All Water Quality Parameters
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("RUNNING PIPELINE FOR ALL PARAMETERS")
print("=" * 80)

print("""
Running the complete pipeline for all three water quality parameters.
The same predictor variables are used across all models:
  X: SWIR22, NDMI, MNDWI, PET

While the response variable changes for each model:
  y1: Total Alkalinity (TA)
  y2: Electrical Conductance (EC)

In [ ]:
# ============================================
# Pipeline Definition + Run All Parameters
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("MODEL WORKFLOW PIPELINE")
print("=" * 80)

# -------------------------------------------------------
# Define Pipeline Function
# -------------------------------------------------------
def run_pipeline(X, y, param_name="Parameter"):
    """
    Complete ML pipeline with anti-overfitting measures.
    """
    print(f"\n{'=' * 60}")
    print(f"PIPELINE: {param_name}")
    print(f"{'=' * 60}")

    # Step 1: Split
    X_train, X_test, y_train, y_test = split_data(X, y)
    print(f"\nStep 1 - Split: Train={X_train.shape[0]:,} | Test={X_test.shape[0]:,}")

    # Step 2: Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    print("Step 2 - Scaled: RobustScaler applied")

    # Step 3: Cross validation
    print("Step 3 - Cross Validation (5-fold):")
    model_cv = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    )
    cv_scores = cross_val_score(
        model_cv, X_train_scaled, y_train,
        cv=5, scoring='r2', n_jobs=-1
    )
    cv_mean = round(float(cv_scores.mean()), 4)
    cv_std  = round(float(cv_scores.std()),  4)
    cv_list = [round(float(s), 4) for s in cv_scores]

    print("  CV R2 scores: " + str(cv_list))
    print("  Mean CV R2:   " + str(cv_mean))
    print("  Std CV R2:    " + str(cv_std))

    if cv_std < 0.05:
        print("  Stable model - low variance across folds!")
    else:
        print("  Warning - high variance across folds!")

    # Step 4: Train
    model = train_model(X_train_scaled, y_train)
    print("\nStep 4 - Trained: " + str(model.n_estimators) + " trees | max_depth=" + str(model.max_depth))

    # Step 5: Evaluate train
    print("\nStep 5 - In-sample evaluation:")
    y_train_pred, r2_train, rmse_train = evaluate_model(
        model, X_train_scaled, y_train, "Train"
    )

    # Step 6: Evaluate test
    print("\nStep 6 - Out-of-sample evaluation:")
    y_test_pred, r2_test, rmse_test = evaluate_model(
        model, X_test_scaled, y_test, "Test"
    )

    # Step 7: Overfitting assessment
    gap = round(float(r2_train) - float(r2_test), 4)
    print("\nStep 7 - Overfitting Assessment:")
    print("  Train R2: " + str(round(float(r2_train), 4)))
    print("  Test R2:  " + str(round(float(r2_test),  4)))
    print("  Gap:      " + str(gap))

    if gap < 0.05:
        overfit_status = "Excellent - minimal overfitting!"
    elif gap < 0.10:
        overfit_status = "Good - acceptable generalization!"
    elif gap < 0.20:
        overfit_status = "Warning - some overfitting detected!"
    else:
        overfit_status = "Poor - significant overfitting!"
    print("  Status:   " + overfit_status)

    if r2_test > 0.75:
        perf_status = "Excellent test performance!"
    elif r2_test > 0.65:
        perf_status = "Good test performance!"
    elif r2_test > 0.50:
        perf_status = "Moderate - consider more features!"
    else:
        perf_status = "Low - needs improvement!"
    print("  Performance: " + perf_status)

    # Step 8: Return results
    results = {
        "Parameter":  param_name,
        "CV_Mean_R2": cv_mean,
        "CV_Std_R2":  cv_std,
        "R2_Train":   round(float(r2_train), 4),
        "RMSE_Train": round(float(rmse_train), 4),
        "R2_Test":    round(float(r2_test), 4),
        "RMSE_Test":  round(float(rmse_test), 4),
        "Gap":        gap,
        "Status":     overfit_status
    }

    return model, scaler, pd.DataFrame([results])

print("Pipeline function defined successfully!")

# -------------------------------------------------------
# Define features and targets
# -------------------------------------------------------
X     = wq_data[['SWIR22', 'NDMI', 'MNDWI', 'PET']]
y_ta  = wq_data['TOTAL_ALKALINITY']
y_ec  = wq_data['ELECTRICAL_CONDUCTANCE']
y_drp = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

print("Feature matrix: " + str(X.shape))
print("Features: " + str(X.columns.tolist()))
print("\n" + "=" * 80)

# -------------------------------------------------------
# Run pipeline for all three parameters
# -------------------------------------------------------
model_ta,  scaler_ta,  results_ta  = run_pipeline(X, y_ta,  "Total Alkalinity")
model_ec,  scaler_ec,  results_ec  = run_pipeline(X, y_ec,  "Electrical Conductance")
model_drp, scaler_drp, results_drp = run_pipeline(X, y_drp, "Dissolved Reactive Phosphorus")

# -------------------------------------------------------
# Final summary
# -------------------------------------------------------
all_results = pd.concat(
    [results_ta, results_ec, results_drp],
    ignore_index=True
)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)
display(all_results)

print("\n" + "=" * 80)
print("MODEL PERFORMANCE INSIGHTS")
print("=" * 80)

for idx, row in all_results.iterrows():
    print("\n" + str(row['Parameter']) + ":")
    print("  CV R2:    " + str(row['CV_Mean_R2']) + " (Std: " + str(row['CV_Std_R2']) + ")")
    print("  Train R2: " + str(row['R2_Train']))
    print("  Test R2:  " + str(row['R2_Test']))
    print("  Gap:      " + str(row['Gap']))
    print("  Status:   " + str(row['Status']))

best_model  = all_results.loc[all_results['R2_Test'].idxmax(), 'Parameter']
worst_model = all_results.loc[all_results['R2_Test'].idxmin(), 'Parameter']
avg_test_r2 = round(float(all_results['R2_Test'].mean()), 4)
avg_gap     = round(float((all_results['R2_Train'] - all_results['R2_Test']).mean()), 4)

print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print("  Best performing model:  " + str(best_model))
print("  Needs most improvement: " + str(worst_model))
print("  Average Test R2:        " + str(avg_test_r2))
print("  Average Train/Test gap: " + str(avg_gap))

if avg_gap < 0.10:
    print("\n  Models generalize well - no significant overfitting!")
else:
    print("\n  Consider further tuning to reduce overfitting!")

print("\n" + "=" * 80)
print("TARGET PERFORMANCE GOALS:")
print("  Train R2:  0.75 - 0.85")
print("  Test R2:   0.65 - 0.75")
print("  Gap:       < 0.10")
print("  CV Std:    < 0.05")
print("=" * 80)
print("PIPELINE COMPLETE - ALL MODELS TRAINED AND EVALUATED!")
print("=" * 80)

Model Training and Evaluation for Each Parameter

In [ ]:
# ============================================
# Model Training and Evaluation for Each Parameter
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("MODEL TRAINING AND EVALUATION FOR EACH PARAMETER")
print("=" * 80)

print("\nIn this step, we apply the complete modeling pipeline to each of the three")
print("selected water quality parameters - Total Alkalinity, Electrical Conductance,")
print("and Dissolved Reactive Phosphorus. The input feature set (X) remains the same")
print("across all three models, while the target variable (y) changes for each parameter.")

print("\nFor every parameter, the run_pipeline() function is executed, which handles")
print("data preprocessing, model training, and both in-sample and out-of-sample")
print("evaluation. This ensures a consistent workflow and allows for a fair comparison")
print("of model performance across different water quality indicators.")

print("\nAnti-overfitting measures applied in each pipeline run:")
print("  1. RobustScaler - handles outliers in environmental data")
print("  2. Tuned Random Forest - max_depth=12, min_samples_leaf=10")
print("  3. 5-fold cross validation - detects overfitting before training")
print("  4. Train/Test gap monitoring - flags significant overfitting")

print("\n" + "=" * 80)

# -------------------------------------------------------
# Check available columns
# -------------------------------------------------------
print("\nAvailable columns in wq_data:")
print(wq_data.columns.tolist())

print("\n" + "=" * 80)

# -------------------------------------------------------
# Define feature set (X) and target variables (y)
# -------------------------------------------------------
X     = wq_data[['SWIR22', 'NDMI', 'MNDWI', 'PET']]
y_TA  = wq_data['TOTAL_ALKALINITY']
y_EC  = wq_data['ELECTRICAL_CONDUCTANCE']
y_DRP = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

print("\nFeature set (X) shape: " + str(X.shape))
print("Features: " + str(X.columns.tolist()))

print("\nTarget variables:")
print("  y_TA  (Total Alkalinity):             " + str(y_TA.shape[0]) + " samples")
print("  y_EC  (Electrical Conductance):        " + str(y_EC.shape[0]) + " samples")
print("  y_DRP (Dissolved Reactive Phosphorus): " + str(y_DRP.shape[0]) + " samples")

print("\n" + "=" * 80)

# -------------------------------------------------------
# Parameter 1: Total Alkalinity
# -------------------------------------------------------
print("\nPARAMETER 1: TOTAL ALKALINITY")
print("-" * 60)
model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")

# -------------------------------------------------------
# Parameter 2: Electrical Conductance
# -------------------------------------------------------
print("\nPARAMETER 2: ELECTRICAL CONDUCTANCE")
print("-" * 60)
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")

# -------------------------------------------------------
# Parameter 3: Dissolved Reactive Phosphorus
# -------------------------------------------------------
print("\nPARAMETER 3: DISSOLVED REACTIVE PHOSPHORUS")
print("-" * 60)
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

print("\n" + "=" * 80)
print("ALL PIPELINES EXECUTED SUCCESSFULLY!")
print("=" * 80)

# -------------------------------------------------------
# Combine all results
# -------------------------------------------------------
all_results = pd.concat(
    [results_TA, results_EC, results_DRP],
    ignore_index=True
)

print("\n" + "=" * 80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 80)
display(all_results)

# -------------------------------------------------------
# Detailed performance analysis
# -------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL PERFORMANCE ANALYSIS")
print("=" * 80)

for idx, row in all_results.iterrows():
    print("\n" + str(row['Parameter']) + ":")
    print("  Training R2:  " + str(round(float(row['R2_Train']), 3)) +
          " | RMSE: " + str(round(float(row['RMSE_Train']), 3)))
    print("  Test R2:      " + str(round(float(row['R2_Test']), 3)) +
          " | RMSE: " + str(round(float(row['RMSE_Test']), 3)))
    print("  CV Mean R2:   " + str(round(float(row['CV_Mean_R2']), 3)) +
          " | Std: "  + str(round(float(row['CV_Std_R2']), 3)))

    r2_gap = float(row['R2_Train']) - float(row['R2_Test'])
    print("  Gap:          " + str(round(r2_gap, 3)))

    # Overfitting assessment
    if r2_gap < 0.05:
        print("  Overfitting:  Excellent - minimal overfitting!")
    elif r2_gap < 0.10:
        print("  Overfitting:  Good - acceptable generalization!")
    elif r2_gap < 0.20:
        print("  Overfitting:  Warning - some overfitting detected!")
    else:
        print("  Overfitting:  Poor - significant overfitting!")

    # Performance assessment
    if float(row['R2_Test']) > 0.75:
        print("  Performance:  Excellent test performance!")
    elif float(row['R2_Test']) > 0.65:
        print("  Performance:  Good test performance!")
    elif float(row['R2_Test']) > 0.50:
        print("  Performance:  Moderate - consider more features!")
    else:
        print("  Performance:  Low - needs improvement!")

# -------------------------------------------------------
# Overall summary
# -------------------------------------------------------
best_model  = all_results.loc[all_results['R2_Test'].idxmax(), 'Parameter']
worst_model = all_results.loc[all_results['R2_Test'].idxmin(), 'Parameter']
avg_test_r2 = round(float(all_results['R2_Test'].mean()), 4)
avg_gap     = round(float(
    (all_results['R2_Train'] - all_results['R2_Test']).mean()
), 4)

print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print("  Best performing model:  " + str(best_model))
print("  Needs most improvement: " + str(worst_model))
print("  Average Test R2:        " + str(avg_test_r2))
print("  Average Train/Test gap: " + str(avg_gap))

if avg_gap < 0.05:
    print("\n  Excellent - minimal overfitting across all models!")
elif avg_gap < 0.10:
    print("\n  Good - models generalize well to unseen data!")
else:
    print("\n  Consider further tuning to reduce overfitting!")

print("\n" + "=" * 80)
print("TARGET PERFORMANCE GOALS:")
print("  Train R2:  0.75 - 0.85")
print("  Test R2:   0.65 - 0.75")
print("  Gap:       < 0.10")
print("  CV Std:    < 0.05")
print("=" * 80)

# -------------------------------------------------------
# Store models and scalers confirmation
# -------------------------------------------------------
print("\nTrained models and scalers saved:")
print("  model_TA,  scaler_TA  - Total Alkalinity")
print("  model_EC,  scaler_EC  - Electrical Conductance")
print("  model_DRP, scaler_DRP - Dissolved Reactive Phosphorus")

print("\n" + "=" * 80)
print("MODEL TRAINING AND EVALUATION COMPLETE!")
print("=" * 80)

Model Performance Summary

In [ ]:
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("MODEL PERFORMANCE SUMMARY + IMPROVEMENT")
print("=" * 80)

print("""
After training and evaluating the models for each water quality parameter,
the individual performance metrics are combined into a single summary table.
This table consolidates the R2 and RMSE values for both in-sample and
out-of-sample evaluations, enabling an easy comparison of model performance
across Total Alkalinity, Electrical Conductance, and Dissolved Reactive
Phosphorus.

To improve R2 above 0.75 without overfitting or underfitting we apply:
  1. More estimators (500) - more trees = better signal capture
  2. Slightly deeper trees (max_depth=15) - captures more patterns
  3. Lower min_samples_leaf (5) - more flexible leaf nodes
  4. min_samples_split=10 - allows more splits on meaningful patterns
  5. max_features=0.7 - uses more features per split
  6. Keep 5-fold CV to monitor overfitting throughout
""")

print("=" * 80)

# -------------------------------------------------------
# Step 1: Define improved model trainer
# Balanced between underfitting and overfitting
# -------------------------------------------------------
def train_model_v2(X_train_scaled, y_train):
    """
    Improved Random Forest with balanced settings.
    More powerful than v1 but still protected from overfitting.
    
    Changes from v1:
      n_estimators: 300  -> 500  (more stable predictions)
      max_depth:     12  ->  15  (captures more patterns)
      min_samples_split: 20 -> 10 (more flexible splitting)
      min_samples_leaf:  10 ->  5 (more flexible leaves)
      max_features:     0.6 -> 0.7 (uses more features)
    """
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    return model

# -------------------------------------------------------
# Step 2: Define improved pipeline
# -------------------------------------------------------
def run_pipeline_v2(X, y, param_name="Parameter"):
    """
    Improved pipeline targeting R2 > 0.75 without overfitting.
    """
    print("\n" + "=" * 60)
    print("IMPROVED PIPELINE: " + param_name)
    print("=" * 60)

    # Split
    X_train, X_test, y_train, y_test = split_data(X, y)
    print("Split: Train=" + str(X_train.shape[0]) + " | Test=" + str(X_test.shape[0]))

    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    print("Scaled: RobustScaler applied")

    # Cross validate first
    print("Cross Validation (5-fold):")
    model_cv = RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )
    cv_scores = cross_val_score(
        model_cv, X_train_scaled, y_train,
        cv=5, scoring='r2', n_jobs=-1
    )
    cv_mean = round(float(cv_scores.mean()), 4)
    cv_std  = round(float(cv_scores.std()),  4)
    print("  CV R2 scores: " + str([round(float(s), 4) for s in cv_scores]))
    print("  Mean CV R2:   " + str(cv_mean))
    print("  Std CV R2:    " + str(cv_std))

    if cv_std < 0.05:
        print("  Stable model - low variance across folds!")
    else:
        print("  Warning - high variance across folds!")

    # Train improved model
    model = train_model_v2(X_train_scaled, y_train)
    print("Trained: " + str(model.n_estimators) + " trees | max_depth=" + str(model.max_depth))

    # Evaluate train
    print("\nIn-sample evaluation:")
    y_train_pred, r2_train, rmse_train = evaluate_model(
        model, X_train_scaled, y_train, "Train"
    )

    # Evaluate test
    print("\nOut-of-sample evaluation:")
    y_test_pred, r2_test, rmse_test = evaluate_model(
        model, X_test_scaled, y_test, "Test"
    )

    # Gap assessment
    gap = round(float(r2_train) - float(r2_test), 4)
    print("\nOverfitting Assessment:")
    print("  Train R2: " + str(round(float(r2_train), 4)))
    print("  Test R2:  " + str(round(float(r2_test),  4)))
    print("  Gap:      " + str(gap))

    if gap < 0.05:
        overfit_status = "Excellent - minimal overfitting!"
    elif gap < 0.10:
        overfit_status = "Good - acceptable generalization!"
    elif gap < 0.20:
        overfit_status = "Warning - some overfitting!"
    else:
        overfit_status = "Poor - significant overfitting!"
    print("  Status:   " + overfit_status)

    if float(r2_test) > 0.75:
        perf_status = "Excellent - target achieved!"
    elif float(r2_test) > 0.65:
        perf_status = "Good - close to target!"
    elif float(r2_test) > 0.50:
        perf_status = "Moderate - needs improvement!"
    else:
        perf_status = "Low - significant improvement needed!"
    print("  Performance: " + perf_status)

    results = {
        "Parameter":  param_name,
        "CV_Mean_R2": cv_mean,
        "CV_Std_R2":  cv_std,
        "R2_Train":   round(float(r2_train),  4),
        "RMSE_Train": round(float(rmse_train), 4),
        "R2_Test":    round(float(r2_test),    4),
        "RMSE_Test":  round(float(rmse_test),  4),
        "Gap":        gap,
        "Status":     overfit_status,
        "Performance": perf_status
    }

    return model, scaler, pd.DataFrame([results])

print("Improved pipeline defined!")
print("=" * 80)

# -------------------------------------------------------
# Step 3: Run improved pipeline for all parameters
# -------------------------------------------------------
print("\nRUNNING IMPROVED PIPELINE FOR ALL PARAMETERS")
print("=" * 80)

X     = wq_data[['SWIR22', 'NDMI', 'MNDWI', 'PET']]
y_TA  = wq_data['TOTAL_ALKALINITY']
y_EC  = wq_data['ELECTRICAL_CONDUCTANCE']
y_DRP = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

model_TA_v2,  scaler_TA_v2,  results_TA_v2  = run_pipeline_v2(X, y_TA,  "Total Alkalinity")
model_EC_v2,  scaler_EC_v2,  results_EC_v2  = run_pipeline_v2(X, y_EC,  "Electrical Conductance")
model_DRP_v2, scaler_DRP_v2, results_DRP_v2 = run_pipeline_v2(X, y_DRP, "Dissolved Reactive Phosphorus")

# -------------------------------------------------------
# Step 4: Combine improved results
# -------------------------------------------------------
results_summary_improved = pd.concat(
    [results_TA_v2, results_EC_v2, results_DRP_v2],
    ignore_index=True
)

print("\n" + "=" * 80)
print("IMPROVED MODEL PERFORMANCE SUMMARY")
print("=" * 80)
display(results_summary_improved)

# -------------------------------------------------------
# Step 5: Compare old vs new
# -------------------------------------------------------
print("\n" + "=" * 80)
print("COMPARISON: ORIGINAL vs IMPROVED")
print("=" * 80)

comparison = pd.DataFrame({
    'Parameter':    all_results['Parameter'],
    'Old_R2_Test':  all_results['R2_Test'],
    'New_R2_Test':  results_summary_improved['R2_Test'],
    'Improvement':  results_summary_improved['R2_Test'] - all_results['R2_Test'],
    'Old_Gap':      all_results['R2_Train'] - all_results['R2_Test'],
    'New_Gap':      results_summary_improved['R2_Train'] - results_summary_improved['R2_Test']
})
display(comparison)

# -------------------------------------------------------
# Step 6: Final summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("PERFORMANCE IMPROVEMENT ANALYSIS")
print("=" * 80)

for idx, row in comparison.iterrows():
    print("\n" + str(row['Parameter']) + ":")
    print("  Old Test R2:  " + str(round(float(row['Old_R2_Test']), 4)))
    print("  New Test R2:  " + str(round(float(row['New_R2_Test']), 4)))
    improvement = round(float(row['Improvement']), 4)
    if improvement > 0:
        print("  Improvement: +" + str(improvement) + " - Better!")
    else:
        print("  Improvement: "  + str(improvement) + " - Needs more tuning!")

    old_gap = round(float(row['Old_Gap']), 4)
    new_gap = round(float(row['New_Gap']), 4)
    print("  Old Gap: " + str(old_gap) + " -> New Gap: " + str(new_gap))

    if new_gap < old_gap:
        print("  Overfitting REDUCED by " + str(round(old_gap - new_gap, 4)))
    elif new_gap == old_gap:
        print("  Overfitting unchanged")
    else:
        print("  Overfitting slightly increased - monitor closely!")

    if float(row['New_R2_Test']) > 0.75:
        print("  TARGET ACHIEVED - R2 above 0.75!")
    else:
        print("  Target not yet reached - consider adding more features")

# Update models to use improved versions
model_TA  = model_TA_v2
model_EC  = model_EC_v2
model_DRP = model_DRP_v2
scaler_TA  = scaler_TA_v2
scaler_EC  = scaler_EC_v2
scaler_DRP = scaler_DRP_v2

print("\n" + "=" * 80)
print("MODELS UPDATED TO IMPROVED VERSIONS!")
print("  model_TA,  scaler_TA  - Total Alkalinity (improved)")
print("  model_EC,  scaler_EC  - Electrical Conductance (improved)")
print("  model_DRP, scaler_DRP - Dissolved Reactive Phosphorus (improved)")
print("\nThese improved models will be used for submission!")
print("=" * 80)

In [ ]:
# ============================================
# Submission - Predictions for Validation Data
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("SUBMISSION - GENERATING PREDICTIONS")
print("=" * 80)

# -------------------------------------------------------
# Step 1: Load all required files
# -------------------------------------------------------
print("\nStep 1: Loading required files...")

# Load submission template
test_file = session.table("SUBMISSION_TEMPLATE").to_pandas()
print("  Submission template: " + str(test_file.shape))
display(test_file.head(5))

# Load validation Landsat features
landsat_val_features = session.table("LANDSAT_FEATURES_VALIDATION").to_pandas()
print("\n  Landsat validation: " + str(landsat_val_features.shape))
display(landsat_val_features.head(5))

# Load validation TerraClimate features
Terraclimate_val_df = session.table("TERRACLIMATE_FEATURES_VALIDATION").to_pandas()
print("\n  TerraClimate validation: " + str(Terraclimate_val_df.shape))
display(Terraclimate_val_df.head(5))

print("\n" + "=" * 80)

# -------------------------------------------------------
# Step 2: Check column names in validation data
# -------------------------------------------------------
print("\nStep 2: Checking column names...")
print("  Landsat val columns:     " + str(landsat_val_features.columns.tolist()))
print("  TerraClimate val columns: " + str(Terraclimate_val_df.columns.tolist()))

print("\n" + "=" * 80)

# -------------------------------------------------------
# Step 3: Consolidate validation features
# Matches original memo structure but with
# uppercase column names to match training data
# -------------------------------------------------------
print("\nStep 3: Consolidating validation features...")

# Map lowercase column names from validation
# to uppercase used in training
col_map_landsat = {
    'nir':    'NIR',
    'green':  'GREEN',
    'swir16': 'SWIR16',
    'swir22': 'SWIR22',
    'NDMI':   'NDMI',
    'MNDWI':  'MNDWI'
}

col_map_terra = {
    'pet': 'PET'
}

# Rename columns to uppercase
landsat_val_features = landsat_val_features.rename(
    columns={k: v for k, v in col_map_landsat.items()
             if k in landsat_val_features.columns}
)
Terraclimate_val_df = Terraclimate_val_df.rename(
    columns={k: v for k, v in col_map_terra.items()
             if k in Terraclimate_val_df.columns}
)

print("  Columns standardized to uppercase!")

# -------------------------------------------------------
# Step 4: Engineer same features as training
# -------------------------------------------------------
print("\nStep 4: Engineering validation features...")

# Convert to numeric
for col in ['NIR', 'GREEN', 'SWIR16', 'SWIR22', 'NDMI', 'MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(
            landsat_val_features[col], errors='coerce'
        )

# Find and convert date column
date_col = None
for col in landsat_val_features.columns:
    if 'DATE' in col.upper() or 'date' in col.lower():
        date_col = col
        break

if date_col:
    landsat_val_features[date_col] = pd.to_datetime(
        landsat_val_features[date_col], dayfirst=True
    )
    print("  Date column found: " + str(date_col))

# New spectral indices
landsat_val_features['NDWI'] = (
    (landsat_val_features['GREEN'] - landsat_val_features['NIR']) /
    (landsat_val_features['GREEN'] + landsat_val_features['NIR'])
)
landsat_val_features['NIR_SWIR_RATIO'] = (
    landsat_val_features['NIR'] /
    landsat_val_features['SWIR22'].replace(0, np.nan)
)
landsat_val_features['SWIR_RATIO'] = (
    landsat_val_features['SWIR16'] /
    landsat_val_features['SWIR22'].replace(0, np.nan)
)
print("  New spectral indices computed!")

# Temporal features
if date_col:
    landsat_val_features['MONTH'] = landsat_val_features[date_col].dt.month
    landsat_val_features['YEAR']  = landsat_val_features[date_col].dt.year

    def get_season(month):
        if month in [12, 1, 2]:
            return 1
        elif month in [3, 4, 5]:
            return 2
        elif month in [6, 7, 8]:
            return 3
        else:
            return 4

    landsat_val_features['SEASON']    = landsat_val_features['MONTH'].apply(get_season)
    landsat_val_features['MONTH_SIN'] = np.sin(
        2 * np.pi * landsat_val_features['MONTH'] / 12
    )
    landsat_val_features['MONTH_COS'] = np.cos(
        2 * np.pi * landsat_val_features['MONTH'] / 12
    )
    print("  Temporal features computed!")

# TerraClimate derived features
Terraclimate_val_df['PET'] = pd.to_numeric(
    Terraclimate_val_df['PET'], errors='coerce'
)
Terraclimate_val_df['PET_SQUARED'] = Terraclimate_val_df['PET'] ** 2
Terraclimate_val_df['PET_LOG']     = np.log1p(
    Terraclimate_val_df['PET'].clip(lower=0)
)

terra_date_col = None
for col in Terraclimate_val_df.columns:
    if 'DATE' in col.upper() or 'date' in col.lower():
        terra_date_col = col
        break

if terra_date_col:
    Terraclimate_val_df[terra_date_col] = pd.to_datetime(
        Terraclimate_val_df[terra_date_col], dayfirst=True
    )
    pet_monthly_mean = Terraclimate_val_df.groupby(
        Terraclimate_val_df[terra_date_col].dt.month
    )['PET'].transform('mean')
    Terraclimate_val_df['PET_MONTHLY_ANOMALY'] = (
        Terraclimate_val_df['PET'] - pet_monthly_mean
    )

if 'LATITUDE' in Terraclimate_val_df.columns:
    Terraclimate_val_df['LOC_CLUSTER'] = pd.cut(
        Terraclimate_val_df['LATITUDE'],
        bins=5, labels=[1, 2, 3, 4, 5]
    ).astype(float)

print("  TerraClimate features computed!")

# -------------------------------------------------------
# Step 5: Build consolidated validation dataframe
# Same structure as original memo but with all features
# -------------------------------------------------------
print("\nStep 5: Building consolidated validation dataframe...")

val_data = pd.DataFrame({
    'Longitude':       landsat_val_features['LONGITUDE'].values
                       if 'LONGITUDE' in landsat_val_features.columns
                       else landsat_val_features['Longitude'].values,
    'Latitude':        landsat_val_features['LATITUDE'].values
                       if 'LATITUDE' in landsat_val_features.columns
                       else landsat_val_features['Latitude'].values,
    'Sample Date':     landsat_val_features[date_col].values
                       if date_col else np.nan,
    # Original bands
    'NIR':             landsat_val_features['NIR'].values,
    'GREEN':           landsat_val_features['GREEN'].values,
    'SWIR16':          landsat_val_features['SWIR16'].values,
    'SWIR22':          landsat_val_features['SWIR22'].values,
    # Original indices
    'NDMI':            landsat_val_features['NDMI'].values,
    'MNDWI':           landsat_val_features['MNDWI'].values,
    # New spectral indices
    'NDWI':            landsat_val_features['NDWI'].values,
    'NIR_SWIR_RATIO':  landsat_val_features['NIR_SWIR_RATIO'].values,
    'SWIR_RATIO':      landsat_val_features['SWIR_RATIO'].values,
    # Temporal features
    'MONTH':           landsat_val_features['MONTH'].values,
    'YEAR':            landsat_val_features['YEAR'].values,
    'SEASON':          landsat_val_features['SEASON'].values,
    'MONTH_SIN':       landsat_val_features['MONTH_SIN'].values,
    'MONTH_COS':       landsat_val_features['MONTH_COS'].values,
    # TerraClimate features
    'PET':             Terraclimate_val_df['PET'].values,
    'PET_SQUARED':     Terraclimate_val_df['PET_SQUARED'].values,
    'PET_LOG':         Terraclimate_val_df['PET_LOG'].values,
    'PET_MONTHLY_ANOMALY': Terraclimate_val_df['PET_MONTHLY_ANOMALY'].values,
    'LOC_CLUSTER':     Terraclimate_val_df['LOC_CLUSTER'].values,
})

# Impute missing values
val_data = val_data.fillna(val_data.median(numeric_only=True))

print("  Consolidated val_data shape: " + str(val_data.shape))
print("\nValidation data preview:")
display(val_data.head(5))

# -------------------------------------------------------
# Step 6: Extract feature matrix for prediction
# Must match EXACTLY the training feature order
# -------------------------------------------------------
print("\nStep 6: Extracting feature matrix...")

feature_cols = [
    'NIR', 'GREEN', 'SWIR16', 'SWIR22',
    'NDMI', 'MNDWI', 'NDWI',
    'NIR_SWIR_RATIO', 'SWIR_RATIO',
    'PET', 'PET_SQUARED', 'PET_LOG',
    'PET_MONTHLY_ANOMALY', 'LOC_CLUSTER',
    'MONTH', 'YEAR', 'SEASON',
    'MONTH_SIN', 'MONTH_COS'
]

submission_val_data = val_data[feature_cols]

print("  Feature matrix shape: " + str(submission_val_data.shape))
display(submission_val_data.head())

# -------------------------------------------------------
# Step 7: Generate predictions
# -------------------------------------------------------
print("\nStep 7: Generating predictions...")

# Scale and predict Total Alkalinity
X_sub_scaled_TA  = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)
print("  Total Alkalinity predicted!")

# Scale and predict Electrical Conductance
X_sub_scaled_EC  = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)
print("  Electrical Conductance predicted!")

# Scale and predict Dissolved Reactive Phosphorus
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)
print("  Dissolved Reactive Phosphorus predicted!")

print("\nPrediction ranges:")
print("  Total Alkalinity:              min=" +
      str(round(float(pred_TA_submission.min()), 2)) +
      " max=" + str(round(float(pred_TA_submission.max()), 2)) +
      " mean=" + str(round(float(pred_TA_submission.mean()), 2)))
print("  Electrical Conductance:        min=" +
      str(round(float(pred_EC_submission.min()), 2)) +
      " max=" + str(round(float(pred_EC_submission.max()), 2)) +
      " mean=" + str(round(float(pred_EC_submission.mean()), 2)))
print("  Dissolved Reactive Phosphorus: min=" +
      str(round(float(pred_DRP_submission.min()), 2)) +
      " max=" + str(round(float(pred_DRP_submission.max()), 2)) +
      " mean=" + str(round(float(pred_DRP_submission.mean()), 2)))

# -------------------------------------------------------
# Step 8: Build submission dataframe
# -------------------------------------------------------
print("\nStep 8: Building submission dataframe...")

submission_df = pd.DataFrame({
    'Longitude':                     test_file['Longitude'].values,
    'Latitude':                      test_file['Latitude'].values,
    'Sample Date':                   test_file['Sample Date'].values,
    'Total Alkalinity':              pred_TA_submission,
    'Electrical Conductance':        pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

print("\nSubmission dataframe preview:")
display(submission_df.head(10))

print("\nSubmission shape: " + str(submission_df.shape))

# -------------------------------------------------------
# Step 9: Save and upload
# -------------------------------------------------------
print("\nStep 9: Saving submission file...")

# Save to CSV
submission_df.to_csv("/tmp/submission.csv", index=False)
print("  Saved to /tmp/submission.csv!")

# Upload to Snowflake workspace
session.sql("""
    PUT file:///tmp/submission.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("  File uploaded to Snowflake workspace!")
print("  Refresh the browser to see the file in the sidebar!")

print("\n" + "=" * 80)
print("SUBMISSION COMPLETE!")
print("=" * 80)
print("""
Next steps:
  1. Download submission.csv from the sidebar
  2. Upload to the EY challenge platform
  3. Check your score on the leaderboard
""")
print("=" * 80)# ============================================
# Submission - Predictions for Validation Data
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("SUBMISSION - GENERATING PREDICTIONS")
print("=" * 80)

# -------------------------------------------------------
# Step 1: Load all required files
# -------------------------------------------------------
print("\nStep 1: Loading required files...")

# Load submission template
test_file = session.table("SUBMISSION_TEMPLATE").to_pandas()
print("  Submission template: " + str(test_file.shape))
display(test_file.head(5))

# Load validation Landsat features
landsat_val_features = session.table("LANDSAT_FEATURES_VALIDATION").to_pandas()
print("\n  Landsat validation: " + str(landsat_val_features.shape))
display(landsat_val_features.head(5))

# Load validation TerraClimate features
Terraclimate_val_df = session.table("TERRACLIMATE_FEATURES_VALIDATION").to_pandas()
print("\n  TerraClimate validation: " + str(Terraclimate_val_df.shape))
display(Terraclimate_val_df.head(5))

print("\n" + "=" * 80)

# -------------------------------------------------------
# Step 2: Check column names in validation data
# -------------------------------------------------------
print("\nStep 2: Checking column names...")
print("  Landsat val columns:     " + str(landsat_val_features.columns.tolist()))
print("  TerraClimate val columns: " + str(Terraclimate_val_df.columns.tolist()))

print("\n" + "=" * 80)

# -------------------------------------------------------
# Step 3: Consolidate validation features
# Matches original memo structure but with
# uppercase column names to match training data
# -------------------------------------------------------
print("\nStep 3: Consolidating validation features...")

# Map lowercase column names from validation
# to uppercase used in training
col_map_landsat = {
    'nir':    'NIR',
    'green':  'GREEN',
    'swir16': 'SWIR16',
    'swir22': 'SWIR22',
    'NDMI':   'NDMI',
    'MNDWI':  'MNDWI'
}

col_map_terra = {
    'pet': 'PET'
}

# Rename columns to uppercase
landsat_val_features = landsat_val_features.rename(
    columns={k: v for k, v in col_map_landsat.items()
             if k in landsat_val_features.columns}
)
Terraclimate_val_df = Terraclimate_val_df.rename(
    columns={k: v for k, v in col_map_terra.items()
             if k in Terraclimate_val_df.columns}
)

print("  Columns standardized to uppercase!")

# -------------------------------------------------------
# Step 4: Engineer same features as training
# -------------------------------------------------------
print("\nStep 4: Engineering validation features...")

# Convert to numeric
for col in ['NIR', 'GREEN', 'SWIR16', 'SWIR22', 'NDMI', 'MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(
            landsat_val_features[col], errors='coerce'
        )

# Find and convert date column
date_col = None
for col in landsat_val_features.columns:
    if 'DATE' in col.upper() or 'date' in col.lower():
        date_col = col
        break

if date_col:
    landsat_val_features[date_col] = pd.to_datetime(
        landsat_val_features[date_col], dayfirst=True
    )
    print("  Date column found: " + str(date_col))

# New spectral indices
landsat_val_features['NDWI'] = (
    (landsat_val_features['GREEN'] - landsat_val_features['NIR']) /
    (landsat_val_features['GREEN'] + landsat_val_features['NIR'])
)
landsat_val_features['NIR_SWIR_RATIO'] = (
    landsat_val_features['NIR'] /
    landsat_val_features['SWIR22'].replace(0, np.nan)
)
landsat_val_features['SWIR_RATIO'] = (
    landsat_val_features['SWIR16'] /
    landsat_val_features['SWIR22'].replace(0, np.nan)
)
print("  New spectral indices computed!")

# Temporal features
if date_col:
    landsat_val_features['MONTH'] = landsat_val_features[date_col].dt.month
    landsat_val_features['YEAR']  = landsat_val_features[date_col].dt.year

    def get_season(month):
        if month in [12, 1, 2]:
            return 1
        elif month in [3, 4, 5]:
            return 2
        elif month in [6, 7, 8]:
            return 3
        else:
            return 4

    landsat_val_features['SEASON']    = landsat_val_features['MONTH'].apply(get_season)
    landsat_val_features['MONTH_SIN'] = np.sin(
        2 * np.pi * landsat_val_features['MONTH'] / 12
    )
    landsat_val_features['MONTH_COS'] = np.cos(
        2 * np.pi * landsat_val_features['MONTH'] / 12
    )
    print("  Temporal features computed!")

# TerraClimate derived features
Terraclimate_val_df['PET'] = pd.to_numeric(
    Terraclimate_val_df['PET'], errors='coerce'
)
Terraclimate_val_df['PET_SQUARED'] = Terraclimate_val_df['PET'] ** 2
Terraclimate_val_df['PET_LOG']     = np.log1p(
    Terraclimate_val_df['PET'].clip(lower=0)
)

terra_date_col = None
for col in Terraclimate_val_df.columns:
    if 'DATE' in col.upper() or 'date' in col.lower():
        terra_date_col = col
        break

if terra_date_col:
    Terraclimate_val_df[terra_date_col] = pd.to_datetime(
        Terraclimate_val_df[terra_date_col], dayfirst=True
    )
    pet_monthly_mean = Terraclimate_val_df.groupby(
        Terraclimate_val_df[terra_date_col].dt.month
    )['PET'].transform('mean')
    Terraclimate_val_df['PET_MONTHLY_ANOMALY'] = (
        Terraclimate_val_df['PET'] - pet_monthly_mean
    )

if 'LATITUDE' in Terraclimate_val_df.columns:
    Terraclimate_val_df['LOC_CLUSTER'] = pd.cut(
        Terraclimate_val_df['LATITUDE'],
        bins=5, labels=[1, 2, 3, 4, 5]
    ).astype(float)

print("  TerraClimate features computed!")

# -------------------------------------------------------
# Step 5: Build consolidated validation dataframe
# Same structure as original memo but with all features
# -------------------------------------------------------
print("\nStep 5: Building consolidated validation dataframe...")

val_data = pd.DataFrame({
    'Longitude':       landsat_val_features['LONGITUDE'].values
                       if 'LONGITUDE' in landsat_val_features.columns
                       else landsat_val_features['Longitude'].values,
    'Latitude':        landsat_val_features['LATITUDE'].values
                       if 'LATITUDE' in landsat_val_features.columns
                       else landsat_val_features['Latitude'].values,
    'Sample Date':     landsat_val_features[date_col].values
                       if date_col else np.nan,
    # Original bands
    'NIR':             landsat_val_features['NIR'].values,
    'GREEN':           landsat_val_features['GREEN'].values,
    'SWIR16':          landsat_val_features['SWIR16'].values,
    'SWIR22':          landsat_val_features['SWIR22'].values,
    # Original indices
    'NDMI':            landsat_val_features['NDMI'].values,
    'MNDWI':           landsat_val_features['MNDWI'].values,
    # New spectral indices
    'NDWI':            landsat_val_features['NDWI'].values,
    'NIR_SWIR_RATIO':  landsat_val_features['NIR_SWIR_RATIO'].values,
    'SWIR_RATIO':      landsat_val_features['SWIR_RATIO'].values,
    # Temporal features
    'MONTH':           landsat_val_features['MONTH'].values,
    'YEAR':            landsat_val_features['YEAR'].values,
    'SEASON':          landsat_val_features['SEASON'].values,
    'MONTH_SIN':       landsat_val_features['MONTH_SIN'].values,
    'MONTH_COS':       landsat_val_features['MONTH_COS'].values,
    # TerraClimate features
    'PET':             Terraclimate_val_df['PET'].values,
    'PET_SQUARED':     Terraclimate_val_df['PET_SQUARED'].values,
    'PET_LOG':         Terraclimate_val_df['PET_LOG'].values,
    'PET_MONTHLY_ANOMALY': Terraclimate_val_df['PET_MONTHLY_ANOMALY'].values,
    'LOC_CLUSTER':     Terraclimate_val_df['LOC_CLUSTER'].values,
})

# Impute missing values
val_data = val_data.fillna(val_data.median(numeric_only=True))

print("  Consolidated val_data shape: " + str(val_data.shape))
print("\nValidation data preview:")
display(val_data.head(5))

# -------------------------------------------------------
# Step 6: Extract feature matrix for prediction
# Must match EXACTLY the training feature order
# -------------------------------------------------------
print("\nStep 6: Extracting feature matrix...")

feature_cols = [
    'NIR', 'GREEN', 'SWIR16', 'SWIR22',
    'NDMI', 'MNDWI', 'NDWI',
    'NIR_SWIR_RATIO', 'SWIR_RATIO',
    'PET', 'PET_SQUARED', 'PET_LOG',
    'PET_MONTHLY_ANOMALY', 'LOC_CLUSTER',
    'MONTH', 'YEAR', 'SEASON',
    'MONTH_SIN', 'MONTH_COS'
]

submission_val_data = val_data[feature_cols]

print("  Feature matrix shape: " + str(submission_val_data.shape))
display(submission_val_data.head())

# -------------------------------------------------------
# Step 7: Generate predictions
# -------------------------------------------------------
print("\nStep 7: Generating predictions...")

# Scale and predict Total Alkalinity
X_sub_scaled_TA  = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)
print("  Total Alkalinity predicted!")

# Scale and predict Electrical Conductance
X_sub_scaled_EC  = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)
print("  Electrical Conductance predicted!")

# Scale and predict Dissolved Reactive Phosphorus
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)
print("  Dissolved Reactive Phosphorus predicted!")

print("\nPrediction ranges:")
print("  Total Alkalinity:              min=" +
      str(round(float(pred_TA_submission.min()), 2)) +
      " max=" + str(round(float(pred_TA_submission.max()), 2)) +
      " mean=" + str(round(float(pred_TA_submission.mean()), 2)))
print("  Electrical Conductance:        min=" +
      str(round(float(pred_EC_submission.min()), 2)) +
      " max=" + str(round(float(pred_EC_submission.max()), 2)) +
      " mean=" + str(round(float(pred_EC_submission.mean()), 2)))
print("  Dissolved Reactive Phosphorus: min=" +
      str(round(float(pred_DRP_submission.min()), 2)) +
      " max=" + str(round(float(pred_DRP_submission.max()), 2)) +
      " mean=" + str(round(float(pred_DRP_submission.mean()), 2)))

# -------------------------------------------------------
# Step 8: Build submission dataframe
# -------------------------------------------------------
print("\nStep 8: Building submission dataframe...")

submission_df = pd.DataFrame({
    'Longitude':                     test_file['Longitude'].values,
    'Latitude':                      test_file['Latitude'].values,
    'Sample Date':                   test_file['Sample Date'].values,
    'Total Alkalinity':              pred_TA_submission,
    'Electrical Conductance':        pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

print("\nSubmission dataframe preview:")
display(submission_df.head(10))

print("\nSubmission shape: " + str(submission_df.shape))

# -------------------------------------------------------
# Step 9: Save and upload
# -------------------------------------------------------
print("\nStep 9: Saving submission file...")

# Save to CSV
submission_df.to_csv("/tmp/submission.csv", index=False)
print("  Saved to /tmp/submission.csv!")

# Upload to Snowflake workspace
session.sql("""
    PUT file:///tmp/submission.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("  File uploaded to Snowflake workspace!")
print("  Refresh the browser to see the file in the sidebar!")

print("\n" + "=" * 80)
print("SUBMISSION COMPLETE!")
print("=" * 80)
print("""
Next steps:
  1. Download submission.csv from the sidebar
  2. Upload to the EY challenge platform
  3. Check your score on the leaderboard
""")
print("=" * 80)